In [0]:
#Config

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)   # fixed seed = same data every run

N_DAYS = 60
START_DATE = pd.Timestamp("2026-07-01")
N_CUSTOMERS = 20000

In [0]:
#Creating the layer schemas

for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{schema}")

In [0]:
#Stores and daily context

stores = pd.DataFrame({
    "store_id": [f"DS0{i}" for i in range(1, 7)],
    "store_name": [f"Dark Store {i}" for i in range(1, 7)],
    "zone": ["North", "North", "South", "South", "East", "West"],
    "base_daily_orders": [320, 280, 350, 260, 300, 290],  # how busy each store's area is
})

dates = pd.date_range(START_DATE, periods=N_DAYS, freq="D")
daily_context = pd.DataFrame({
    "date": dates.date,
    "is_weekend": dates.dayofweek >= 5,
    "is_rainy": rng.random(N_DAYS) < 0.20,   # roughly 1 rainy day in 5
})

In [0]:
#SKUs

categories = [
    # name, number of SKUs, is_perishable, typical price (INR)
    ("Fruits & Vegetables", 60, True, 60),
    ("Dairy & Eggs", 35, False, 70),
    ("Bakery", 20, False, 50),
    ("Snacks", 50, False, 40),
    ("Beverages", 40, False, 60),
    ("Staples", 35, False, 150),
    ("Personal Care", 30, False, 180),
    ("Household", 30, False, 120),
]

rows, n = [], 1
for name, count, perishable, typical_price in categories:
    for _ in range(count):
        rows.append({
            "sku_id": f"SKU{n:04d}",
            "category": name,
            "is_perishable": perishable,
            "unit_price": int(max(10, rng.lognormal(np.log(typical_price), 0.4))),
            "shelf_life_days": int(rng.integers(2, 6)) if perishable else 180,
            "popularity": float(rng.pareto(1.5) + 1),   # heavy tail: few SKUs dominate
        })
        n += 1
skus = pd.DataFrame(rows)

In [0]:
#Customers

w = stores["base_daily_orders"] / stores["base_daily_orders"].sum()
signup = START_DATE + pd.to_timedelta(rng.integers(-400, N_DAYS, N_CUSTOMERS), unit="D")

customers = pd.DataFrame({
    "customer_id": [f"C{i:06d}" for i in range(1, N_CUSTOMERS + 1)],
    "signup_date": signup,
    "home_store_id": rng.choice(stores["store_id"], N_CUSTOMERS, p=w),
})
customers["signup_date"] = customers["signup_date"].dt.date

In [0]:
#Save to Bronze

def save_bronze(pdf, name):
    sdf = spark.createDataFrame(pdf)
    sdf.write.mode("overwrite").saveAsTable(f"workspace.bronze.{name}")
    print(f"{name}: {sdf.count()} rows")

save_bronze(stores, "stores")
save_bronze(skus, "skus")
save_bronze(customers, "customers")
save_bronze(daily_context, "daily_context")

In [0]:
%sql
--Check

SELECT category, COUNT(*) AS skus, ROUND(AVG(unit_price)) AS avg_price
FROM workspace.bronze.skus
GROUP BY category
ORDER BY skus DESC;

In [0]:
#Demand settings

HOURS = np.arange(6, 24)   # stores operate 06:00-23:59
HOUR_WEIGHTS = np.array([0.5, 1.5, 2.5, 3, 3, 4, 5, 5, 4, 3.5, 4, 5, 6.5, 8.5, 9.5, 9, 6, 3])
HOUR_P = HOUR_WEIGHTS / HOUR_WEIGHTS.sum()   # lunch peak and a bigger dinner peak
assert len(HOURS) == len(HOUR_WEIGHTS)

WEEKEND_MULT = 1.15   # weekends are busier
RAIN_MULT = 1.10      # rain increases orders slightly

In [0]:
#who orders(customer pools)

cust_ids = customers["customer_id"].values
cust_weight = rng.gamma(0.7, 1.0, N_CUSTOMERS) + 0.05   # heavy tail: a few customers order a lot
cust_signup = pd.to_datetime(customers["signup_date"]).values

# customers grouped by their home store
home_idx = {s: np.where(customers["home_store_id"].values == s)[0] for s in stores["store_id"]}

In [0]:
#Generate orders, day by day and store by store

frames = []
for day, ctx in zip(dates, daily_context.itertuples()):
    for s in stores.itertuples():
        mult = (WEEKEND_MULT if ctx.is_weekend else 1.0) * (RAIN_MULT if ctx.is_rainy else 1.0)
        n = rng.poisson(s.base_daily_orders * mult)

        # only customers of this store who have already signed up can order
        pool = home_idx[s.store_id]
        pool = pool[cust_signup[pool] <= day.to_datetime64()]
        p = cust_weight[pool] / cust_weight[pool].sum()

        chosen = rng.choice(pool, n, p=p)
        hour = rng.choice(HOURS, n, p=HOUR_P)
        seconds = hour * 3600 + rng.integers(0, 3600, n)

        frames.append(pd.DataFrame({
            "store_id": s.store_id,
            "customer_id": cust_ids[chosen],
            "placed_ts": day + pd.to_timedelta(seconds, unit="s"),
        }))

orders_raw = pd.concat(frames, ignore_index=True).sort_values("placed_ts").reset_index(drop=True)
orders_raw.insert(0, "order_id", [f"ORD{i:07d}" for i in range(1, len(orders_raw) + 1)])

In [0]:
#Basket size, promise time, and the generator's private plan

n_items = np.minimum(1 + rng.poisson(3.0, len(orders_raw)), 12)   # about 4 lines per order
orders_raw["promised_minutes"] = np.where(n_items <= 4, 10, 15)   # bigger baskets get a longer promise

# n_items is the generator's private knowledge (not in the raw feed); module 3 uses it
order_plan = pd.DataFrame({"order_id": orders_raw["order_id"], "n_items": n_items})

orders = orders_raw[["order_id", "customer_id", "store_id", "placed_ts", "promised_minutes"]]
assert orders["order_id"].is_unique
print(len(orders), "orders")

In [0]:
#save to bronze

save_bronze(orders, "orders")

In [0]:
%sql
--demand curve check

SELECT HOUR(placed_ts) AS hr, COUNT(*) AS orders
FROM workspace.bronze.orders
GROUP BY 1 ORDER BY 1;

In [0]:
%sql
--rain effect check
SELECT c.is_rainy,
       COUNT(DISTINCT c.date) AS days,
       ROUND(COUNT(*) / COUNT(DISTINCT c.date)) AS avg_orders_per_day
FROM workspace.bronze.orders o
JOIN workspace.bronze.daily_context c ON DATE(o.placed_ts) = c.date
GROUP BY c.is_rainy;

In [0]:
#Availability grid (with the planted stockout patterns)

S, K, D = len(stores), len(skus), N_DAYS
store_pos = {s: i for i, s in enumerate(stores["store_id"])}

avail = np.ones((S, K, D, 24), dtype=bool)   # avail[store, sku, day, hour] = True if in stock
pop_pct = skus["popularity"].rank(pct=True).values   # 1.0 = most popular SKU
is_dairy = (skus["category"] == "Dairy & Eggs").values
is_fv = (skus["category"] == "Fruits & Vegetables").values

for si, store_id in enumerate(stores["store_id"]):
    # every SKU can have an outage on any day; popular SKUs run out more often
    q = np.tile(0.04 + 0.10 * pop_pct[:, None], (1, D))   # chance of an outage that day
    starts = rng.integers(6, 21, (K, D))                  # hour the outage starts
    lens = rng.integers(1, 7, (K, D))                     # outage length in hours

    # planted patterns
    if store_id == "DS05":            # understocks fruit & veg
        q[is_fv] = 0.50
        lens[is_fv] = rng.integers(3, 8, (is_fv.sum(), D))
    elif store_id == "DS02":          # overstocks fruit & veg (shows up as wastage later)
        q[is_fv] = 0.01
    else:
        q[is_fv] = 0.10
    if store_id == "DS04":            # dairy restocking gap at the evening peak
        n_d = is_dairy.sum()
        q[is_dairy] = 0.75
        starts[is_dairy] = rng.integers(17, 21, (n_d, D))
        lens[is_dairy] = rng.integers(3, 7, (n_d, D))

    for ki, di in np.argwhere(rng.random((K, D)) < q):
        avail[si, ki, di, starts[ki, di]: starts[ki, di] + lens[ki, di]] = False

In [0]:
#Check the patterns are in the grid

for si, store_id in enumerate(stores["store_id"]):
    a = avail[si][:, :, 6:24]
    print(store_id, "all:", round(1 - a.mean(), 4),
          "| dairy:", round(1 - a[is_dairy].mean(), 4),
          "| F&V:", round(1 - a[is_fv].mean(), 4))

In [0]:
#Order lines (which SKUs each order contains)

qty_choices = np.array([1, 2, 3])
qty_p = np.array([0.70, 0.22, 0.08])
sku_p = (skus["popularity"] / skus["popularity"].sum()).values
n_items_arr = order_plan["n_items"].values

order_idx = np.repeat(np.arange(len(orders)), n_items_arr)
sku_idx = np.concatenate([rng.choice(K, size=k, replace=False, p=sku_p) for k in n_items_arr])

lines = pd.DataFrame({
    "order_idx": order_idx,
    "sku_idx": sku_idx,
    "qty_ordered": rng.choice(qty_choices, len(order_idx), p=qty_p),
})
oi = lines["order_idx"].values
ki_arr = lines["sku_idx"].values
placed = orders["placed_ts"]

lines["order_id"] = orders["order_id"].values[oi]
lines["sku_id"] = skus["sku_id"].values[ki_arr]
lines["unit_price"] = skus["unit_price"].values[ki_arr]

sp = orders["store_id"].map(store_pos).values[oi]                              # store position
dy = (placed.dt.normalize() - START_DATE).dt.days.values[oi]                   # day index
hr = placed.dt.hour.values[oi]                                                 # hour of day

oos = ~avail[sp, ki_arr, dy, hr]      # True = item was out of stock when ordered
print(len(lines), "order lines;", round(oos.mean() * 100, 2), "% unavailable")

In [0]:
#What happens to missing items (split, substitute, or remove)

NEIGHBOR = {"DS01": "DS02", "DS02": "DS01", "DS03": "DS04", "DS04": "DS03", "DS05": "DS06", "DS06": "DS05"}
neighbor_pos = np.array([store_pos[NEIGHBOR[s]] for s in stores["store_id"]])

P_SPLIT = 0.45        # knob: chance an eligible order is split across two stores
P_SUBSTITUTE = 0.55   # knob: chance a missing item is replaced by a similar one

# 1) can the missing item ship from the neighbouring store instead?
nb_stock = avail[neighbor_pos[sp], ki_arr, dy, hr]
can_move = oos & nb_stock
n_move = np.bincount(oi, weights=can_move.astype(float), minlength=len(orders))
eligible = (n_move > 0) & (n_move < n_items_arr)     # something moves AND something stays
is_split = eligible & (rng.random(len(orders)) < P_SPLIT)
moved = can_move & is_split[oi]

# 2) other missing items: substitute with a similar in-stock SKU, or remove the item
rest_oos = oos & ~moved
substituted = rest_oos & (rng.random(len(lines)) < P_SUBSTITUTE)
substitute_sku = np.full(len(lines), None, dtype=object)
cat = skus["category"].values
sku_ids_arr = skus["sku_id"].values
for i in np.where(substituted)[0]:
    candidates = np.where((cat == cat[ki_arr[i]]) & avail[sp[i], :, dy[i], hr[i]])[0]
    if len(candidates) == 0:
        substituted[i] = False            # nothing suitable to substitute with
    else:
        substitute_sku[i] = sku_ids_arr[rng.choice(candidates)]
removed = rest_oos & ~substituted

# 3) build the tables (kept in memory for now)
status = np.where(moved, "fulfilled_split",
         np.where(substituted, "substituted",
         np.where(removed, "removed", "fulfilled")))

order_items = lines[["order_id", "sku_id", "qty_ordered", "unit_price"]].copy()
order_items["qty_fulfilled"] = np.where(removed, 0, lines["qty_ordered"].values)
order_items["item_status"] = status
order_items["sub_order_id"] = np.where(moved, lines["order_id"] + "-S2", lines["order_id"] + "-S1")
order_items["substitute_sku_id"] = substitute_sku

primary = pd.DataFrame({
    "sub_order_id": orders["order_id"] + "-S1", "order_id": orders["order_id"],
    "store_id": orders["store_id"], "sub_order_seq": 1,
})
second = pd.DataFrame({
    "sub_order_id": orders["order_id"][is_split] + "-S2", "order_id": orders["order_id"][is_split],
    "store_id": [NEIGHBOR[s] for s in orders["store_id"][is_split]], "sub_order_seq": 2,
})
sub_orders = (pd.concat([primary, second], ignore_index=True)
                .sort_values(["order_id", "sub_order_seq"]).reset_index(drop=True))

# generator's private notes for the next module (cancellations depend on these)
order_plan["n_oos"] = np.bincount(oi, weights=oos.astype(float), minlength=len(orders)).astype(int)
order_plan["is_split"] = is_split

In [0]:
#Checks

print("orders:", len(orders), "| order_items:", len(order_items), "| sub_orders:", len(sub_orders))
print("split share of orders:", round(is_split.mean() * 100, 2), "%")
print(order_items["item_status"].value_counts(normalize=True).round(4).to_string())

assert order_items.groupby("order_id").size().eq(n_items_arr).all()
assert sub_orders["sub_order_id"].is_unique
assert set(order_items["sub_order_id"]).issubset(set(sub_orders["sub_order_id"]))
print("all checks passed")

In [0]:
#Per-order setup

N = len(orders)
placed = orders["placed_ts"]
placed_v = placed.values
hour = placed.dt.hour.values
day_idx = (placed.dt.normalize() - START_DATE).dt.days.values
rainy = daily_context["is_rainy"].values[day_idx]
shortage = np.isin(hour, [19, 20, 21])            # planted: dinner rider shortage
busy = np.isin(hour, [12, 13, 19, 20, 21])        # lunch and dinner peaks
rain_mult = np.where(rainy, rng.uniform(1.3, 1.5, N), 1.0)   # rain slows everything by 30-50%

# lines physically picked per shipment, and unavailable items per order
picked = order_items["item_status"].isin(["fulfilled", "substituted", "fulfilled_split"])
n_pick_by_so = order_items[picked].groupby("sub_order_id").size()
order_pos = pd.Series(np.arange(N), index=orders["order_id"])
n_pick1 = orders["order_id"].add("-S1").map(n_pick_by_so).fillna(0).astype(int).values
n_oos = order_plan["n_oos"].values

In [0]:
#Shipment simulator

SHORT_ASSIGN = 1.5              # knob: mean minutes to assign a rider in the dinner shortage (0.5 otherwise)
COORD_LO, COORD_HI = 3.0, 6.0   # knob: extra minutes a split order loses coordinating two stores

def simulate_leg(idx, t_pick_start, n_pick, n_oos_leg, ride_mult=1.0):
    """Simulates one shipment. All times are minutes after the order was placed."""
    m = len(idx)
    rain_m = rain_mult[idx]
    short = shortage[idx]

    # picking and packing (missing items take extra time to handle)
    pick_dur = 0.9 + 0.25 * n_pick + rng.exponential(0.3, m) + 0.6 * n_oos_leg
    t_packed = t_pick_start + pick_dur + 0.4 + rng.exponential(0.2, m)

    # rider assignment: slower in the dinner shortage and in rain; sometimes reassigned
    assign_delay = rng.exponential(np.where(short, SHORT_ASSIGN, 0.5), m) * rain_m
    t_assigned = t_pick_start + assign_delay
    reassign = rng.random(m) < np.where(short | rainy[idx], 0.12, 0.05)
    t_assigned_2 = np.where(reassign, t_assigned + 0.8 + rng.exponential(0.8, m), np.nan)
    t_assigned_final = np.where(reassign, t_assigned_2, t_assigned)

    # rider travels to the store, picks up, rides to the customer
    travel = (1.0 + rng.exponential(0.8, m)) * rain_m * np.where(short, 1.4, 1.0)
    t_arrive = t_assigned_final + travel
    t_pickup = np.maximum(t_packed, t_arrive) + 0.2
    ride = rng.gamma(4.0, 0.85, m) * rain_m * ride_mult
    return dict(idx=idx, pick_start=t_pick_start, pick_dur=pick_dur, packed=t_packed,
                assigned=t_assigned, reassign=reassign, assigned_2=t_assigned_2,
                arrive=t_arrive, pickup=t_pickup, delivered=t_pickup + ride)

In [0]:
# shipment 1: from the store the customer ordered from
t_accept = rng.exponential(0.3, N)
pick_wait = rng.exponential(0.4, N) * np.where(busy, 1.5, 1.0)
leg1 = simulate_leg(np.arange(N), t_accept + pick_wait, n_pick1, n_oos)

# shipment 2: only for split orders, from the neighbouring store
split_idx = np.where(order_plan["is_split"].values)[0]
m2 = len(split_idx)
t_split = leg1["pick_start"][split_idx] + rng.uniform(0.2, 0.7, m2) * leg1["pick_dur"][split_idx]
n_pick2 = pd.Series(orders["order_id"].values[split_idx]).add("-S2").map(n_pick_by_so).fillna(0).astype(int).values
t_ps2 = (t_split + rng.uniform(COORD_LO, COORD_HI, m2)
         + rng.exponential(0.5, m2) * np.where(busy[split_idx], 1.5, 1.0))
leg2 = simulate_leg(split_idx, t_ps2, n_pick2, np.zeros(m2), ride_mult=1.35)

# an order is delivered when its last shipment arrives
total_min = leg1["delivered"].copy()
total_min[split_idx] = np.maximum(leg1["delivered"][split_idx], leg2["delivered"])
first_pickup = leg1["pickup"].copy()
first_pickup[split_idx] = np.minimum(leg1["pickup"][split_idx], leg2["pickup"])
promised = orders["promised_minutes"].values
is_split = order_plan["is_split"].values

In [0]:
#Cancellations

n_removed = (order_items.groupby("order_id")["item_status"]
             .apply(lambda s: (s == "removed").sum()).reindex(orders["order_id"]).values)
removed_frac = n_removed / order_plan["n_items"].values
all_removed = n_removed == order_plan["n_items"].values
rider_wait = leg1["pickup"] - leg1["packed"]

p_cancel = (0.010 + 0.010 * rainy + 0.03 * (n_removed >= 1) + 0.20 * (removed_frac >= 0.5)
            + 0.12 * (total_min > promised + 4) + 0.06 * is_split)
p_cancel = np.clip(p_cancel, 0, 0.6)
p_cancel[all_removed] = 1.0                 # nothing left to deliver
cancelled = rng.random(N) < p_cancel

# a cancellation can only happen before the first pickup
lo = t_accept + 0.2
hi = first_pickup - 0.1
cancel_min = np.where(cancelled, rng.uniform(lo, hi), np.nan)

reason = np.select(
    [removed_frac >= 0.5, rider_wait > 3, total_min > promised + 4],
    ["items_unavailable", "rider_unavailable", "delay_customer_cancelled"],
    default="customer_changed_mind")
reason = np.where(cancelled, reason, None)

In [0]:
#Event-building helpers

oid = orders["order_id"].values
allN = np.arange(N)
none_ = np.full(N, None, dtype=object)

def rider_ids(store_ids, m):
    return pd.Series(store_ids).astype(str) + "-R" + pd.Series(rng.integers(1, 41, m)).astype(str).str.zfill(2)

def frame(idx, sub_ids, etype, minutes, sku=None, rider=None, why=None):
    m = len(idx)
    return pd.DataFrame({
        "order_pos": idx,
        "order_id": oid[idx],
        "sub_order_id": sub_ids,
        "event_type": etype,
        "t_min": minutes,
        "sku_id": sku if sku is not None else np.full(m, None, dtype=object),
        "rider_id": rider if rider is not None else np.full(m, None, dtype=object),
        "cancel_reason": why if why is not None else np.full(m, None, dtype=object),
    })

def leg_events(leg, sub_suffix, store_ids):
    idx = leg["idx"]; m = len(idx)
    sid = pd.Series(oid[idx]).add(sub_suffix).values
    r1 = rider_ids(store_ids, m).values
    # a reassigned order gets a different rider from the same store
    r2 = np.where(leg["reassign"], (pd.Series(r1).str[-2:].astype(int).values % 40 + 1), 0)
    r2 = pd.Series(store_ids).astype(str).values + "-R" + pd.Series(r2).astype(str).str.zfill(2).values
    final_r = np.where(leg["reassign"], r2, r1)
    rm = leg["reassign"]
    return [
        frame(idx, sid, "picking_started", leg["pick_start"]),
        frame(idx, sid, "packing_completed", leg["packed"]),
        frame(idx, sid, "rider_assigned", leg["assigned"], rider=r1),
        frame(idx[rm], sid[rm], "rider_assigned", leg["assigned_2"][rm], rider=r2[rm]),
        frame(idx, sid, "rider_arrived_at_store", leg["arrive"], rider=final_r),
        frame(idx, sid, "order_picked_up", leg["pickup"], rider=final_r),
        frame(idx, sid, "order_delivered", leg["delivered"], rider=final_r),
    ]

In [0]:
#Assemble all events, apply cancellations, finalize

parts = [
    frame(allN, none_, "order_placed", np.zeros(N)),
    frame(allN, none_, "order_accepted", t_accept),
]
parts += leg_events(leg1, "-S1", orders["store_id"].values)
sec_stores = np.array([NEIGHBOR[s] for s in orders["store_id"].values[split_idx]])
parts += leg_events(leg2, "-S2", sec_stores)

# item_unavailable, item_substituted, order_split
st_arr = order_items["item_status"].values
oo_pos = order_pos.reindex(order_items["order_id"]).values
unav = np.isin(st_arr, ["fulfilled_split", "substituted", "removed"])
u_idx = np.where(unav)[0]
u_ord = oo_pos[u_idx]
u_moved = st_arr[u_idx] == "fulfilled_split"
t_split_full = np.full(N, np.nan)
t_split_full[split_idx] = t_split
u_t = np.where(u_moved,
               t_split_full[u_ord] - rng.uniform(0.02, 0.15, len(u_idx)),
               leg1["pick_start"][u_ord] + rng.uniform(0.15, 0.9, len(u_idx)) * leg1["pick_dur"][u_ord])
parts.append(frame(u_ord, pd.Series(oid[u_ord]).add("-S1").values, "item_unavailable", u_t,
                   sku=order_items["sku_id"].values[u_idx]))
s_mask = st_arr[u_idx] == "substituted"
parts.append(frame(u_ord[s_mask], pd.Series(oid[u_ord[s_mask]]).add("-S1").values, "item_substituted",
                   u_t[s_mask] + 0.2 + rng.exponential(0.3, s_mask.sum()),
                   sku=order_items["sku_id"].values[u_idx][s_mask]))
parts.append(frame(split_idx, pd.Series(oid[split_idx]).add("-S2").values, "order_split", t_split))

ev = pd.concat(parts, ignore_index=True)

# cancelled orders: drop everything after the cancel time, then add the cancel event
cm = cancel_min[ev["order_pos"].values]
ev = ev[~(ev["t_min"].values > cm)]
c_idx = np.where(cancelled)[0]
ev = pd.concat([ev, frame(c_idx, none_[c_idx], "order_cancelled", cancel_min[c_idx], why=reason[c_idx])],
               ignore_index=True)

# minutes -> timestamps, sort, give ids
secs = np.round(ev["t_min"].values * 60).astype(int)
ev["event_ts"] = placed_v[ev["order_pos"].values] + pd.to_timedelta(secs, unit="s").values
ev = ev.sort_values(["event_ts", "order_pos"], kind="stable").reset_index(drop=True)
ev.insert(0, "event_id", [f"EVT{i:08d}" for i in range(1, len(ev) + 1)])
order_events = ev[["event_id", "order_id", "sub_order_id", "event_type", "event_ts",
                   "sku_id", "rider_id", "cancel_reason"]]

# cancelled orders deliver nothing
mask_c = order_items["order_id"].isin(set(oid[cancelled]))
order_items.loc[mask_c, "qty_fulfilled"] = 0
order_items.loc[mask_c, "item_status"] = "cancelled"
order_plan["is_cancelled"] = cancelled
print("events built:", len(order_events))

In [0]:
#Checks

late = total_min > promised
ok = ~cancelled
print(order_events["event_type"].value_counts().to_string())
print("\ncancel rate %:", round(cancelled.mean() * 100, 2),
      "| rainy vs dry:", round(cancelled[rainy].mean() * 100, 2), round(cancelled[~rainy].mean() * 100, 2),
      "| split vs normal:", round(cancelled[is_split].mean() * 100, 2), round(cancelled[~is_split].mean() * 100, 2))
print("late rate % (not cancelled): rainy vs dry:",
      round(late[ok & rainy].mean() * 100, 2), round(late[ok & ~rainy].mean() * 100, 2))
print("late rate % by hour:")
print((pd.Series(late[ok]).groupby(hour[ok]).mean() * 100).round(1).to_string())
print("avg minutes placed->delivered, split vs normal:",
      round(total_min[is_split].mean(), 2), round(total_min[~is_split].mean(), 2))
print("avg minutes packed->pickup, dinner shortage vs other hours:",
      round(rider_wait[shortage].mean(), 2), round(rider_wait[~shortage].mean(), 2))
print("cancel reasons:\n" + pd.Series(reason[cancelled]).value_counts().to_string())

dl = order_events[order_events["event_type"] == "order_delivered"]
cc = order_events[order_events["event_type"] == "order_cancelled"]
assert set(dl["order_id"]).isdisjoint(set(cc["order_id"]))     # no order is both delivered and cancelled
assert cc["order_id"].is_unique
n_sub = sub_orders.groupby("order_id").size()
n_del = dl.groupby("order_id").size().reindex(n_sub.index).fillna(0)
not_cancelled = ~n_sub.index.isin(set(oid[cancelled]))
assert (n_del[not_cancelled] == n_sub[not_cancelled]).all()    # every shipment delivered
print("integrity checks passed")

In [0]:
#Units sold per store, SKU, day, and hour

sku_pos = {s: i for i, s in enumerate(skus["sku_id"])}

line_ord = order_pos.reindex(order_items["order_id"]).values      # which order each line belongs to
st = order_items["item_status"].values
qty = order_items["qty_ordered"].values
prim_store = orders["store_id"].map(store_pos).values[line_ord]
line_day = day_idx[line_ord]
line_hr = hour[line_ord]
sku_orig = order_items["sku_id"].map(sku_pos).values
sub_pos = order_items["substitute_sku_id"].map(sku_pos).values

sold = np.zeros((S, K, D, 24), dtype=np.int32)

# normal lines: sold from the ordering store
m1 = st == "fulfilled"
np.add.at(sold, (prim_store[m1], sku_orig[m1], line_day[m1], line_hr[m1]), qty[m1])

# substituted lines: the substitute SKU is what actually left the store
m2 = st == "substituted"
np.add.at(sold, (prim_store[m2], sub_pos[m2].astype(int), line_day[m2], line_hr[m2]), qty[m2])

# split lines: shipped from the neighbouring store
m3 = st == "fulfilled_split"
np.add.at(sold, (neighbor_pos[prim_store[m3]], sku_orig[m3], line_day[m3], line_hr[m3]), qty[m3])

assert (sold[~avail] == 0).all()      # nothing is ever sold while out of stock

In [0]:
#Inventory snapshots (start of each hour, 06:00 to 23:00)

# popular SKUs hold more stock; bigger stores hold more
lam = (2 + 12 * pop_pct)[None, :] * (stores["base_daily_orders"].values / 300)[:, None]
buffer = rng.poisson(lam[:, :, None, None], size=(S, K, D, 24))
on_hand = np.where(avail, sold + 1 + buffer, 0).astype(np.int16)    # 0 whenever out of stock

si, ki, di, hi = np.meshgrid(np.arange(S), np.arange(K), np.arange(D), np.arange(6, 24), indexing="ij")
inventory_snapshots = pd.DataFrame({
    "store_id": stores["store_id"].values[si.ravel()],
    "sku_id": skus["sku_id"].values[ki.ravel()],
    "snapshot_ts": START_DATE + pd.to_timedelta(di.ravel() * 24 + hi.ravel(), unit="h"),
    "on_hand_qty": on_hand[:, :, :, 6:24].ravel(),
})

In [0]:
#Fruit and vegetable wastage

fv_idx = np.where(is_fv)[0]
sold_fv = sold.sum(axis=3)[:, fv_idx, :]            # units sold per store, F&V SKU, day

# how much each store orders relative to what it sells (planted patterns)
cover_by_store = {"DS02": 2.20, "DS05": 1.03}       # DS02 overstocks, DS05 understocks
cover = np.array([cover_by_store.get(s, 1.25) for s in stores["store_id"]])

received = np.ceil(sold_fv * cover[:, None, None] * rng.uniform(0.95, 1.10, sold_fv.shape)).astype(int)
received = np.maximum(received, rng.integers(0, 3, sold_fv.shape))    # minimum delivery even on slow days
received = np.maximum(received, sold_fv)                              # can't sell what never arrived

leftover = received - sold_fv
shelf = skus["shelf_life_days"].values[fv_idx]
wasted = rng.binomial(leftover, 1.0 / shelf[None, :, None])           # short shelf life = more spoilage

si2, fi2, di2 = np.meshgrid(np.arange(S), np.arange(len(fv_idx)), np.arange(D), indexing="ij")
wastage_log = pd.DataFrame({
    "store_id": stores["store_id"].values[si2.ravel()],
    "sku_id": skus["sku_id"].values[fv_idx][fi2.ravel()],
    "date": dates.date[di2.ravel()],
    "qty_received": received.ravel(),
    "qty_wasted": wasted.ravel(),
})

In [0]:
#Checks

print("snapshots:", len(inventory_snapshots), "| wastage rows:", len(wastage_log))
print("zero-stock share of snapshots %:", round((inventory_snapshots["on_hand_qty"] == 0).mean() * 100, 2))
print("DS04 dairy zero-stock share (should match Cell 16):",
      round((on_hand[3][is_dairy][:, :, 6:24] == 0).mean(), 4))

w = wastage_log.groupby("store_id")[["qty_received", "qty_wasted"]].sum()
print("F&V wastage % of received, by store:")
print((w["qty_wasted"] / w["qty_received"] * 100).round(1).to_string())

assert not inventory_snapshots.duplicated(["store_id", "sku_id", "snapshot_ts"]).any()
print("checks passed")

In [0]:
#Reorder thinning

# who had a bad first experience? (cancelled, late, or partial)
bad_exp = cancelled | (total_min > promised) | (n_removed >= 1)
is_first = ~orders["customer_id"].duplicated().values        # orders are sorted by time
first_ts = orders.loc[is_first].set_index("customer_id")["placed_ts"]
bad_first = pd.Series(bad_exp[is_first], index=orders.loc[is_first, "customer_id"].values)

cust_first_ts = orders["customer_id"].map(first_ts)
cust_bad = orders["customer_id"].map(bad_first).astype(bool).values
within_7d = ((orders["placed_ts"] > cust_first_ts) &
             (orders["placed_ts"] <= cust_first_ts + pd.Timedelta(days=7))).values

P_DROP = 0.45      # knob: share of follow-up orders (within 7 days) lost after a bad first order
drop = cust_bad & within_7d & (rng.random(N) < P_DROP)
keep_ids = set(orders["order_id"].values[~drop])

# thinned copies of every table (originals stay untouched, so this cell is safe to re-run)
orders_final = orders[~drop].reset_index(drop=True)
sub_orders_final = sub_orders[sub_orders["order_id"].isin(keep_ids)].reset_index(drop=True)
order_items_final = order_items[order_items["order_id"].isin(keep_ids)].reset_index(drop=True)
events_final = order_events[order_events["order_id"].isin(keep_ids)].reset_index(drop=True)
print("orders dropped:", int(drop.sum()), "| orders kept:", len(orders_final))

# check the pattern: 7-day reorder rate after a bad vs good first order
o = orders_final[["customer_id", "placed_ts"]].copy()
o["first_ts"] = o["customer_id"].map(first_ts)
o["reorder7"] = (o["placed_ts"] > o["first_ts"]) & (o["placed_ts"] <= o["first_ts"] + pd.Timedelta(days=7))
r = o.groupby("customer_id")["reorder7"].any()
end_ts = orders["placed_ts"].max()
eligible_c = first_ts[first_ts <= end_ts - pd.Timedelta(days=7)].index   # a full 7 days of follow-up exists
bad_c = bad_first.reindex(eligible_c)
print("customers with a bad first order:", int(bad_c.sum()), "of", len(bad_c))
print("7-day reorder rate %, bad first vs good first:",
      round(r.reindex(bad_c[bad_c].index).mean() * 100, 1),
      round(r.reindex(bad_c[~bad_c].index).mean() * 100, 1))

In [0]:
#Make raw event feed messy

ev = events_final.copy()
n_ev = len(ev)

# normal ingestion: a few seconds after the event; about 1% arrive 30 minutes to 6 hours late
lag_s = rng.exponential(3.0, n_ev)
late_ev = rng.random(n_ev) < 0.01
lag_s = np.where(late_ev, rng.uniform(1800, 21600, n_ev), lag_s)
ev["ingested_at"] = ev["event_ts"] + pd.to_timedelta(np.round(lag_s).astype(int), unit="s").values

# about 0.5% of rider events lose their rider_id
rider_types = ev["event_type"].isin(["rider_assigned", "rider_arrived_at_store",
                                     "order_picked_up", "order_delivered"]).values
null_rider = rider_types & (rng.random(n_ev) < 0.005)
ev.loc[null_rider, "rider_id"] = None

# about 1.5% of events are delivered twice (same event_id, ingested again a bit later)
dup = ev[rng.random(n_ev) < 0.015].copy()
dup["ingested_at"] = dup["ingested_at"] + pd.to_timedelta(rng.integers(5, 900, len(dup)), unit="s").values

# the raw feed arrives in ingestion order, so late events show up out of order
order_events_raw = (pd.concat([ev, dup], ignore_index=True)
                      .sort_values("ingested_at", kind="stable").reset_index(drop=True))
inventory_snapshots["on_hand_qty"] = inventory_snapshots["on_hand_qty"].astype("int32")

print("raw events:", len(order_events_raw), "| duplicated rows:", len(dup),
      "| late arrivals:", int(late_ev.sum()), "| null rider ids:", int(null_rider.sum()))
print("distinct event_ids:", order_events_raw["event_id"].nunique())

In [0]:
#Save to bronze

def save_bronze(pdf, name, chunk_rows=200_000):
    """Writes a pandas DataFrame to a Bronze table in chunks, keeping each transfer small."""
    full = f"workspace.bronze.{name}"
    for i, start in enumerate(range(0, len(pdf), chunk_rows)):
        writer = (spark.createDataFrame(pdf.iloc[start:start + chunk_rows])
                       .write.mode("overwrite" if i == 0 else "append"))
        if i == 0:
            writer = writer.option("overwriteSchema", "true")
        writer.saveAsTable(full)
    print(f"{name}: {spark.table(full).count()} rows")

# generator-only columns must not leak into the raw data, so we re-save these two without them
save_bronze(stores.drop(columns=["base_daily_orders"]), "stores")
save_bronze(skus.drop(columns=["popularity"]), "skus")

# new or changed tables
save_bronze(orders_final, "orders")
save_bronze(sub_orders_final, "sub_orders")
save_bronze(order_items_final, "order_items")
save_bronze(order_events_raw, "order_events")
save_bronze(inventory_snapshots, "inventory_snapshots")
save_bronze(wastage_log, "wastage_log")

In [0]:
%sql
--checks

SELECT COUNT(*) AS rows_total,
       COUNT(DISTINCT event_id) AS distinct_events,
       SUM(CASE WHEN rider_id IS NULL AND event_type IN
             ('rider_assigned','rider_arrived_at_store','order_picked_up','order_delivered')
           THEN 1 ELSE 0 END) AS null_rider_events,
       SUM(CASE WHEN unix_timestamp(ingested_at) - unix_timestamp(event_ts) > 1800
           THEN 1 ELSE 0 END) AS late_rows
FROM workspace.bronze.order_events;

In [0]:
%sql
SELECT 'stores' AS tbl, COUNT(*) AS n FROM workspace.bronze.stores
UNION ALL SELECT 'skus', COUNT(*) FROM workspace.bronze.skus
UNION ALL SELECT 'customers', COUNT(*) FROM workspace.bronze.customers
UNION ALL SELECT 'daily_context', COUNT(*) FROM workspace.bronze.daily_context
UNION ALL SELECT 'orders', COUNT(*) FROM workspace.bronze.orders
UNION ALL SELECT 'sub_orders', COUNT(*) FROM workspace.bronze.sub_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM workspace.bronze.order_items
UNION ALL SELECT 'order_events', COUNT(*) FROM workspace.bronze.order_events
UNION ALL SELECT 'inventory_snapshots', COUNT(*) FROM workspace.bronze.inventory_snapshots
UNION ALL SELECT 'wastage_log', COUNT(*) FROM workspace.bronze.wastage_log;